In [1]:
import os
os.chdir(r"F:\DataScience")

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import openpyxl


In [3]:
# 3. Load dataset
df = sns.load_dataset('mpg')   # mpg dataset

# %%
# 4. Keep required columns
df = df[['mpg','cylinders','displacement','horsepower','weight','acceleration',
         'model_year','origin','name']].copy()

# %%
# 5. Handle missing values
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())
df = df.dropna().reset_index(drop=True)

# %%
# 6. Encode 'origin' (string → numeric)
df['origin'] = df['origin'].astype(str).map({'usa': 1, 'europe': 2, 'japan': 3})

# Remove name from X (keep for reference only)
# (name is string and cannot be used in Ridge)
df_no_name = df.drop(columns=['name']).copy()

# %%
# 7. Add Serial Number (SR)
df_no_name.insert(0, 'SR', range(1, len(df_no_name)+1))


In [5]:
# 8. Define X and y
X = df_no_name[['cylinders','displacement','horsepower','weight',
                'acceleration','model_year','origin']]
y = df_no_name['mpg']

# %%
# 9. Train-test split (ONE line)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)
train_df = df_no_name.loc[X_train.index].reset_index(drop=True)
test_df = df_no_name.loc[X_test.index].reset_index(drop=True)
print(X.head())
print(y.head())

   cylinders  displacement  horsepower  weight  acceleration  model_year  \
0          8         307.0       130.0    3504          12.0          70   
1          8         350.0       165.0    3693          11.5          70   
2          8         318.0       150.0    3436          11.0          70   
3          8         304.0       150.0    3433          12.0          70   
4          8         302.0       140.0    3449          10.5          70   

   origin  
0       1  
1       1  
2       1  
3       1  
4       1  
0    18.0
1    15.0
2    18.0
3    16.0
4    17.0
Name: mpg, dtype: float64


In [6]:
# 10. Scale features (important for Ridge)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

feature_names = X.columns.tolist()
scales = scaler.scale_
means = scaler.mean_

In [7]:
# 11. Train Ridge model
alpha = 1.0   # change alpha to show ridge effect
ridge = Ridge(alpha=alpha, random_state=42)
ridge.fit(X_train_s, y_train)

# Coefficients (scaled space)
coef_scaled = ridge.coef_
intercept_scaled = ridge.intercept_

# Convert coefficients to original scale
coef_orig = coef_scaled / scales
intercept_orig = intercept_scaled - np.sum((means / scales) * coef_scaled)

coeff_df = pd.DataFrame({
    'Feature': feature_names,
    'Coef_Scaled': coef_scaled,
    'Coef_OriginalScale': coef_orig
})
coeff_df = coeff_df[['Feature','Coef_Scaled','Coef_OriginalScale']]

# Add intercept row
intercept_row = pd.DataFrame([{
    'Feature': 'Intercept',
    'Coef_Scaled': intercept_scaled,
    'Coef_OriginalScale': intercept_orig
}])
coeff_df_with_intercept = pd.concat([intercept_row, coeff_df], ignore_index=True)


In [14]:
# 12. Predictions and residuals
y_pred_train = ridge.predict(X_train_s)
y_pred_test = ridge.predict(X_test_s)

resid_train = y_train.values - y_pred_train
resid_test = y_test.values - y_pred_test
#print(y_pred_train)
#print(y_pred_test)
#print(resid_train)
#print(resid_test)

In [15]:
# 13. Evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae = mean_absolute_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

results_df = pd.DataFrame({
    'Metric': ['RMSE','MAE','R2'],
    'Value': [rmse, mae, r2]
})


In [16]:
# 14. Training and Testing Predictions Table
train_result = train_df.copy()
train_result['Actual_mpg'] = y_train.values
train_result['Pred_mpg'] = np.round(y_pred_train, 3)
train_result['Residual'] = np.round(resid_train, 3)

test_result = test_df.copy()
test_result['Actual_mpg'] = y_test.values
test_result['Pred_mpg'] = np.round(y_pred_test, 3)
test_result['Residual'] = np.round(resid_test, 3)


In [17]:
# 15. Residual summary
resid_summary = pd.DataFrame({
    'Dataset': ['Train','Test'],
    'Count': [len(resid_train), len(resid_test)],
    'RMSE': [
        np.sqrt(mean_squared_error(y_train, y_pred_train)),
        rmse
    ],
    'MAE': [
        mean_absolute_error(y_train, y_pred_train),
        mae
    ],
    'R2': [
        r2_score(y_train, y_pred_train),
        r2
    ]
})


In [18]:
# 16. Ridge Explanation Sheet
ridge_explain = pd.DataFrame({
    'Concept': [
        'Ridge Regression (L2 regularization)',
        'Goal',
        'Key idea',
        'Alpha (λ)',
        'Effect on coefficients',
        'Difference from Lasso',
        'Why scale features'
    ],
    'Explanation': [
        'Ridge adds an L2 penalty (sum of squared coefficients) to OLS.',
        'Reduce overfitting and stabilize coefficients (especially with multicollinearity).',
        'Minimize SSE + alpha * sum(beta_j^2).',
        'Higher alpha → stronger shrinkage → coefficients move closer to zero.',
        'L2 penalty shrinks coefficients but does NOT force them to be exactly zero.',
        'Lasso can set coefficients to 0 (feature selection), Ridge cannot.',
        'Scaling required because penalty depends on coefficient magnitude.'
    ]
})


In [19]:
# 17. Export Everything to Excel
with pd.ExcelWriter('Ridge_MPG_All.xlsx', engine='openpyxl') as writer:
    df_no_name.to_excel(writer, sheet_name='Raw Data', index=False)
    train_df.to_excel(writer, sheet_name='Training Data', index=False)
    test_df.to_excel(writer, sheet_name='Testing Data', index=False)
    results_df.to_excel(writer, sheet_name='Model Summary', index=False)
    coeff_df_with_intercept.to_excel(writer, sheet_name='Coefficients', index=False)
    train_result.to_excel(writer, sheet_name='Training Predictions', index=False)
    test_result.to_excel(writer, sheet_name='Testing Predictions', index=False)
    resid_summary.to_excel(writer, sheet_name='Residual Summary', index=False)
    ridge_explain.to_excel(writer, sheet_name='Ridge Explanation', index=False)

print("✅ Excel file 'Ridge_MPG_All.xlsx' created successfully.")
print("Model Summary:"); print(results_df)


✅ Excel file 'Ridge_MPG_All.xlsx' created successfully.
Model Summary:
  Metric     Value
0   RMSE  3.049836
1    MAE  2.374107
2     R2  0.837929
